# Get to Know a Dataset: Handwritten Digit Classification with Neural Cultures

This notebook is a 101-level guided tour of the **Handwritten Digit Classification with Neural Cultures** dataset from Cortical Labs.

The dataset contains approximately 85 GB of time-series electrophysiological recordings and associated metadata across 52 experimental recordings (MNIST runs). Human iPSC-derived neuronal networks were interfaced with the CL1 biocomputing platform during a spatio-temporal handwritten digit recognition task.

- [Dataset documentation](https://github.com/Cortical-Labs/handwritten-digit-neural-cultures)
- [Registry of Open Data on AWS landing page (available after launch)](https://registry.opendata.aws/handwritten-digit-neural-cultures/)
- [Associated bioRxiv preprint](https://www.biorxiv.org/content/10.64898/2026.08.10.743829v1)


## Q: How is the dataset organized?

The bucket root contains the CSV manifest `file_names.csv`. Each recording is stored beneath a parent directory identified by `Chip ID` and a child directory identified by `Date`. Each recording directory contains 1,000 numbered HDF5 chunks.

```text
/dataset_root/
|-- file_names.csv
|-- [Chip_ID_1]/[Date_1]/
|   |-- 0001.h5
|   |-- ...
|   `-- 1000.h5
`-- [Chip_ID_N]/[Date_N]/
    |-- 0001.h5
    |-- ...
    `-- 1000.h5
```

The manifest maps each run to its system, chip, recording date, cell type, spike count, and classification/control metrics.


## Python requirements

Install these packages using the method appropriate for your environment:

```text
boto3>=1.38
pandas>=2.2
matplotlib>=3.8
cl-sdk
```


In [ ]:
import os
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory

os.environ["CL_SDK_VISUALISATION"] = "0"

import boto3
import matplotlib.pyplot as plt
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
from cl import RecordingView

pd.set_option("display.max_columns", None)


## Configure public S3 access

Replace `REPLACE_AFTER_AWS_BUCKET_IS_CREATED` after AWS confirms the sponsored account is linked and the final public bucket has been created. Unsigned requests are used for the public dataset.


In [ ]:
BUCKET_NAME = "REPLACE_AFTER_AWS_BUCKET_IS_CREATED"
MANIFEST_KEY = "ael3906_Suppl. Excel_seq1_v1_2.csv"

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))


The following cell lists objects and prefixes at the bucket root after the bucket name is available.


In [ ]:
if BUCKET_NAME.startswith("REPLACE_"):
    print("Set BUCKET_NAME after the public S3 bucket is created.")
else:
    response = s3.list_objects_v2(Bucket=BUCKET_NAME, Delimiter="/", MaxKeys=100)
    print("Objects:", [item["Key"] for item in response.get("Contents", [])])
    print("Prefixes:", [item["Prefix"] for item in response.get("CommonPrefixes", [])])


## Q: What data formats are present?

The dataset uses two formats:

1. **CSV** — the master manifest and data dictionary index. It contains 15 fields describing each recording and its associated classification and control metrics.
2. **HDF5 (`.h5`)** — chunked electrophysiological time-series files. HDF5 supports hierarchical scientific data and efficient access to large numerical arrays.

The recommended interface for CL1 recordings is [`cl.RecordingView`](https://docs.corticallabs.com/cl/analysis), provided by the Cortical Labs SDK and analysis toolkit. It gives convenient access to recording attributes, samples, spikes, stimulation events, and data streams when present. Its `file` attribute exposes the underlying PyTables file for lower-level access when required.


## Q: How can I load the manifest?

For development, this notebook first checks for the manifest in the repository root. If it is not present locally and the public bucket name has been configured, it downloads the manifest directly from S3.


In [ ]:
local_manifest = Path("..") / MANIFEST_KEY

if local_manifest.exists():
    manifest = pd.read_csv(
        local_manifest,
        dtype={"Sys-ID": "string", "Chip ID": "string", "Date": "string", "Cell Type": "string"},
    )
elif not BUCKET_NAME.startswith("REPLACE_"):
    manifest_object = s3.get_object(Bucket=BUCKET_NAME, Key=MANIFEST_KEY)
    manifest = pd.read_csv(
        BytesIO(manifest_object["Body"].read()),
        dtype={"Sys-ID": "string", "Chip ID": "string", "Date": "string", "Cell Type": "string"},
    )
else:
    raise RuntimeError(
        "Place the manifest in the repository root or configure BUCKET_NAME."
    )

print(f"Recording rows: {len(manifest)}")
manifest.head()


In [ ]:
print("Manifest fields:")
for field in manifest.columns:
    print(f"- {field}")

manifest.info()


## A first visualization

This chart shows how the 52 manifest rows are distributed across the recorded `Cell Type` categories. The field combines physical architecture and cellular lineage as recorded in the manifest.


In [ ]:
cell_type_counts = manifest["Cell Type"].value_counts(dropna=False).sort_values()

ax = cell_type_counts.plot(
    kind="barh",
    figsize=(9, 5),
    color="#2563EB",
    title="Recordings by Cell Type",
)
ax.set_xlabel("Number of recordings")
ax.set_ylabel("Cell Type")
plt.tight_layout()
plt.show()


## Q: How can I inspect a recording chunk?

The next cell selects the first manifest row, constructs the documented key for `0001.h5`, downloads it to a temporary local file, and opens it with `cl.RecordingView`. The context manager closes the underlying PyTables file automatically.


In [ ]:
example = manifest.dropna(subset=["Chip ID", "Date"]).iloc[0]
example_key = f"{example['Chip ID']}/{example['Date']}/0001.h5"
print("Example HDF5 key:", example_key)

def inspect_recording(recording_path):
    with RecordingView(recording_path) as recording:
        print("Attributes:", recording.attributes)
        print("Samples:", None if recording.samples is None else recording.samples.shape)
        print("Spikes:", None if recording.spikes is None else len(recording.spikes))
        print("Stimulations:", None if recording.stims is None else len(recording.stims))
        print("Data streams:", None if recording.data_streams is None else list(recording.data_streams.keys()))
        print("Underlying PyTables file:", recording.file)

if BUCKET_NAME.startswith("REPLACE_"):
    print("Configure BUCKET_NAME to download and inspect this object.")
else:
    with TemporaryDirectory() as temporary_directory:
        local_recording = Path(temporary_directory) / "0001.h5"
        s3.download_file(BUCKET_NAME, example_key, str(local_recording))
        inspect_recording(local_recording)


## Q: What is one question these data can help answer?

One introductory question is: **How do neuronal spike counts and the `Real` classification accuracy vary across the recorded cell-type categories?**

The following cells use only fields documented in the manifest. They summarize each category and create a scatter plot. This is an exploratory comparison, not a causal analysis; architecture, cellular lineage, recording conditions, and decoder choices may be associated with one another.


In [ ]:
analysis = manifest.copy()
analysis["Num Spikes"] = pd.to_numeric(analysis["Num Spikes"], errors="coerce")
analysis["Real"] = pd.to_numeric(analysis["Real"], errors="coerce")

summary = (
    analysis.groupby("Cell Type", dropna=False)
    .agg(
        recordings=("Sys-ID", "size"),
        median_spikes=("Num Spikes", "median"),
        median_real_accuracy=("Real", "median"),
    )
    .sort_values("median_real_accuracy", ascending=False)
)

summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for cell_type, group in analysis.dropna(subset=["Num Spikes", "Real"]).groupby("Cell Type"):
    ax.scatter(
        group["Num Spikes"],
        group["Real"],
        label=cell_type,
        alpha=0.8,
        s=55,
    )

ax.set_xscale("log")
ax.set_xlabel("Number of spikes (log scale)")
ax.set_ylabel("Real classification accuracy")
ax.set_title("Spike count and classification accuracy by Cell Type")
ax.legend(title="Cell Type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Community challenge

**Challenge:** Develop and validate a comparison of neural architectures and cellular lineages that separates genuine network computation from randomized baselines, stimulus artifacts, and temporal leakage.

A useful starting point would be to:

1. Compare `Real`, `Random`, `Real with Stim`, and `Random with Stim`.
2. Contrast `Time Bin Accuracy Holdout over Trials` with `Time Bin Accuracy Holdout over Time`.
3. Examine whether apparent performance differences remain after accounting for `Num Spikes`.
4. Report results separately by `Cell Type` and use trial-based validation where appropriate.
5. Extend the manifest-level analysis with the documented `cl.RecordingView` and `cl.analysis` metrics, using the underlying PyTables interface only when lower-level access is required.

The associated preprint provides the scientific context and discusses architecture, network dynamics, decoder choice, artifact control, and trial-based cross-validation.


## Citation and licence

If you use this dataset, please cite:

> Loeffler, A., Habibollahi, F., Abu-Bonsrah, K. D., Azadi, A., Desouza, C., Chan, H. W., Nishi, Y., Zhou, J., Doensen, F., Yamamoto, H., Watmuff, B., & Kagan, B. J. (2026). *Handwritten Digit classification with neural cultures is influenced by neural architecture, network dynamics, and decoding methods*. bioRxiv, 2026.08.10.743829. https://doi.org/10.64898/2026.08.10.743829

The dataset is licensed under the [Creative Commons Attribution-NonCommercial 4.0 International licence](https://creativecommons.org/licenses/by-nc/4.0/).
